<a id="top"></a>
# Accessing Roman Catalogs: Leveraging Ancillary Tables and Visualization to Apply Data Quality Cuts
***


## Learning Goals
By the end of this tutorial, you will:

- Understand how to launch and programatically control [MAST Aladin](https://mast-aladin.readthedocs.io/en/latest/index.html) to view targets on the sky from within a notebook, including overlaying a set of targets and observation footprints on the sky.
- Understand how to use the [MAST catalog schema browser](https://mast.stsci.edu/schema_browser/#/) to discover ancillary tables and 
- Understand how to "decode" catalog bitflags, aided by a bitflag "helper" class and the bitflag definition documentation (or database flag definition tables).


### Table of Contents

1. [Introduction](#Introduction)
1. [Imports](#Imports)
1. [Identification and quality checks on candidate stars in the Galactic Bulge](#Main)
   1. [Identifying candidate stars in the Galactic Bulge from PanSTARRS catalogs](#Query)
   2. [Launching MAST-Aladin to inspect targets & outliers](#MASTAladin)
   3. [Examining flag information in the PanSTARRS catalogs](#FlagChecks)
   4. [Refining the sample with data quality cuts](#RefiningSample)
1. [Additional Resources](#Additional-Resources)
1. [About This Notebook](#About-this-Notebook)


## Introduction

This notebook demonstrates how to leverage catalog queries together with [MAST Aladin](https://mast-aladin.readthedocs.io/en/latest/index.html) (to view objects on the sky) and the [MAST catalog schemas](https://mast.stsci.edu/schema_browser/#/) to examine a selected sample, investigate data quality issues, and identify quality flags that can be used to refine the sample for analysis.

For demonstration purposes (as Roman catalogs are not yet available), PanSTARRS will be used as the example dataset. Similar to Roman (particularly the WFI [multiband photometric source catalogs](https://roman-docs.stsci.edu/data-handbook/wfi-data-levels-and-products#WFIDataLevelsandProducts-Level4)),  PanSTARRS features a rich set of ancillary tables containing data quality and processing flags --- providing a reasonable analog to demonstrate MAST data access and visualization tools that will be available for use with Roman.

This tutorial will demonstrate how to query candidate stars in the Galactic Bulge (from PanSTARRS, in the current version), view the targets in MAST Aladin, and then examine data quality flag information from ancillary tables to determine how to exclude data artifacts and sperrious sources from the analysis.

## Imports

This tutorial makes use of the following packages: 
- [*astroquery.mast Catalogs, Observations*](https://astroquery.readthedocs.io/en/latest/mast/mast.html) for querying the MAST catalog database and observation holdings
- [*astropy.coordinates SkyCoord*](https://docs.astropy.org/en/stable/coordinates/index.html) for defining sky coordinates
- [*astropy.units*](https://docs.astropy.org/en/stable/units/index.html) to specify angular units
- [*numpy*](https://numpy.org/doc/stable/) for numerical calculations
- [*matplotlib.pyplot*](https://matplotlib.org/stable/) for plotting catalog data
- [*mast_aladin*](github.com/spacetelescope/mast-aladin) to view targets on the sky
- [*sidecar Sidecar*](https://github.com/jupyter-widgets/jupyterlab-sidecar) to display MAST-Aladin alongside the notebook
- *time* to enable time delays when flipping through objects with MAST-Aladin
- [*astropy.table Table*](https://docs.astropy.org/en/stable/table/index.html) to construct tables from flag values
- A simple "flag helper class" *PanSTARRSFlagHelper* to retrieve definitions (from a table via a [MAST TAP service](http://mast.stsci.edu/vo-tap/)), represent, and help parse data quality flags encoded as bitflags for easier readibility.

In [ ]:
from astroquery.mast import Catalogs, Observations
from astropy.coordinates import SkyCoord
from astropy.table import Table
import astropy.units as u
import numpy as np
from mast_aladin import MastAladin
from sidecar import Sidecar
import matplotlib.pyplot as plt
from flag_helper import PanSTARRSFlagHelper
import time

************

<a id="Main"></a>
## Identification and quality checks on candidate stars in the Galactic Bulge

In this notebook, we will construct a color-magnitude diagram ($r-i$ vs $z$) of stars in the Milky Way bulge near the Galactic Center, from which specific categories of stars can be selected for further analysis.

(In the interest of simplicity, this example will omit crossmatching to Gaia to leverage Gaia-derived distances to exclude fore/background objects.)

<a id="Query"></a>
### Identifying candidate stars in the Galactic Bulge from PanSTARRS catalogs

#### Querying the PanSTARRS catalog

We will begin by querying the PanSTARRS DR2 "mean" table using [`astroquery.mast`](https://astroquery.readthedocs.io/en/latest/mast/mast_catalog.html), near the Galactic Center, with additional quality cuts. Our selection criteria are:
* Located within 0.1 degree of the Galactic center
* Have available $r$, $i$, $z$ magnitudes (i.e., magnitude > -999, the placeholder used for [missing values in the database](https://outerspace.stsci.edu/display/PANSTARRS/PS1+MeanObject+table+fields)). 

We first obtain the sky coordinates for the Galactic center using astropy.

In [ ]:
# Use the Astropy SkyCoord name resolver to get 
# the position of the galactic center in the ICRS frame
gc = SkyCoord.from_name("Galactic center")
gc

Based on PanSTARRS documentation (from [`astroquery.mast`](https://catalogs.mast.stsci.edu/docs/panstarrs.html) and the [PanSTARRS DR2 documentation](https://outerspace.stsci.edu/spaces/PANSTARRS/pages/298812351/PS1+Source+extraction+and+catalogs)), we see the `MeanObject` catalog provides the relevant photometric information (in the columns `rMeanPSFMag`, `iMeanPSFMag`, `zMeanPSFMag`), as well as position information (RA/Dec).


Using the `Catalogs` module of `astroquery.mast`, 
we query the PanSTARRS `MeanObject` table with these constraints.

In [ ]:
# Query the PanSTARRS DR2 Mean catalog aroung the 
# Galactic center, with available z,r,i magnitudes
sample = Catalogs.query_criteria(
    catalog="Panstarrs",
    data_release="dr2",
    table="mean",
    coordinates=gc,
    radius=0.1*u.deg,
    rMeanPSFMag=[("gt", -999)],   # missing magnitudes are -999
    iMeanPSFMag=[("gt", -999)],   # missing magnitudes are -999
    zMeanPSFMag=[("gt", -999)],   # missing magnitudes are -999
)

Next, to distinguish stars (point sources) from extended objects, we will make a cut using the PSF and Kron magnitudes in the `i` band, as point sources typically have `iMeanPSFMAG-iMeanKronMag <= 0.05` (as recommended [in the PanSTARRS documentation](https://outerspace.stsci.edu/display/PANSTARRS/How+to+separate+stars+and+galaxies)).

In [ ]:
sample = sample[
    sample['iMeanPSFMag']-sample['iMeanKronMag'] <= 0.05
]

Finally, we will sort our sample by descending "objID" for consistent ordering.

In [ ]:
sample.sort("objID", reverse=True)

Our resulting sample has 2906 candidate stars:

In [ ]:
sample

#### Visualizing the candidate stars

Next, we plot a preliminary $r-i$ vs $z$ color-magnitude diagram of our candidate stars in the Galactic Bulge.

In [ ]:
f, ax = plt.subplots()
f.set_size_inches((4, 4))
ax.scatter(
    sample['rMeanPSFMag']-sample['iMeanPSFMag'],
    sample['zMeanPSFMag'], 
    s=5, lw=0, 
    alpha=0.5
)

ax.set_xlabel("r-i, PSF Mag")
ax.set_ylabel("z, PSF Mag")

# Invert y axis limits to plot fainter -> brighter
ax.set_ylim(ax.get_ylim()[::-1])

We see a lot of objects in our preliminary sample that are surprisingly red or blue in $r-i$. This suggests there may be data quality cuts that need to be made for our final sample.

To investigate, we first define a color cut to roughly select these very blue or red candidates, where objects with $r-i > 3$ or $<0$ are flagged as potentially suspect (overplotted in red below).

In [ ]:
f, ax = plt.subplots()
f.set_size_inches((4, 4))

# Plot full preliminary sample
ax.scatter(
    sample['rMeanPSFMag']-sample['iMeanPSFMag'],
    sample['zMeanPSFMag'], 
    s=5, lw=0, 
    alpha=0.2
)

# Rough suspect target color cut
whsuspect = (
    (sample['rMeanPSFMag']-sample['iMeanPSFMag'] > 3)
) | (
    (sample['rMeanPSFMag']-sample['iMeanPSFMag'] < 0)
) 

# Overplot potentially suspect targets
ax.scatter(
    (sample['rMeanPSFMag']-sample['iMeanPSFMag'])[whsuspect],
    sample['zMeanPSFMag'][whsuspect], 
    s=5, lw=0, 
    alpha=0.5,
    color='red'
)


ax.set_xlabel("r-i, PSF Mag")
ax.set_ylabel("z, PSF Mag")

ax.set_ylim(ax.get_ylim()[::-1])

Overall, this selection includes 65 (potentially) suspect objects.

In [ ]:
len(sample[whsuspect])

-----

<a id="MASTAladin"></a>
### Launching MAST-Aladin to inspect targets & outliers

We'll now launch MAST-Aladin to the side of our notebook, to visually examine these suspect targets over a sky map of PanSTARRS imaging to hunt for clues regarding what type of quality issues might be at play with these very red/blue objects.

In [ ]:
# Launch MAST-Aladin
with Sidecar(title="Aladin Viewer", anchor="split-right"):
    mast_aladin = MastAladin()
    display(mast_aladin)

As we're working with PanSTARRS data in the galactic bulge, we will also change the base survey imaging to the PanSTARRS1 imaging, and center the viewport at the center of the GBTDS imaging (with a 5 degree field of view).

(See the documentation for [ipyaladin](https://cds-astro.github.io/ipyaladin/index.html), on which MAST-Aladin is built, for examples of how to set the [sky survey displayed](https://cds-astro.github.io/ipyaladin/_collections/notebooks/05_Display_a_MOC.html) and changing [the center and FOV](https://cds-astro.github.io/ipyaladin/_collections/notebooks/02_Base_Commands.html)).

In [ ]:
# Change base imaging survey
mast_aladin.survey = "P/PanSTARRS/DR1/color-i-r-g"

# Change viewport location and FOV (in degrees)
mast_aladin.target = gc
mast_aladin.fov = 0.1   # FOV in degrees

#### Adding targets to MAST-Aladin


We then plot the full set of objects as an overlay to MAST-Aladin, for reference.

Note that we must specify the columns of our sample table the contain the RA/Dec information (`raMean` and `decMean`).

In [ ]:
mast_aladin.add_table(
    sample,
    ra_field="raMean",
    dec_field="decMean",
    name="Sample",
    shape="circle",
    color="yellow",
)

By going into the "Overlays" menu (upper left, 3 "layers" stacked symbol), this table overlay can be turned off and back on --- which can be helpful when investigating the suspect targets.

Next we add the potentially suspect candidates as another overlay table.
<!-- Next we add the full set of objects as an overlay to MAST-Aladin, for reference. -->

In [ ]:
mast_aladin.add_table(
    sample[whsuspect],
    ra_field="raMean",
    dec_field="decMean",
    name="Outliers",
    shape="circle",
    color="cyan",
)

#### Adding observation footprints to MAST-Aladin

It's also possible to add footprints corresponding to individual observations (or reduced mosaic tiles) 
to MAST-Aladin, using those observations' associated `s_region` metadata. 

In this case, all objects in our sample fall within a single PanSTARRS tile per band. The tile boundaries can be seen by zooming out.

(This visualization of where objects fall within images can be useful when investigating whether entire images or regions of images are driving data quality issues.)

First, we query for all PanSTARRS images overlapping our cone search region.

In [ ]:
ps_obs = Observations.query_criteria(
    obs_collection="PS1",
    coordinates=gc,
    radius=0.1*u.deg,
)

We then add these image footprints to MAST-Aladin as overlays.

In [ ]:
colors = ['green', 'blue', 'red', 'yellow', 'orange']
for i in range(len(ps_obs)):
    mast_aladin.add_graphic_overlay_from_stcs(
        ps_obs['s_region'][i], name=ps_obs['filters'][i], 
        color=colors[i]
    )

#### Setting the Viewport to center a target

In addition to panning and zooming in MAST-Aladin, and then clicking on the region circle to obtain information on a specific object (the full table row), it is also possible to programatically change the viewport center and field of view (FOV) to go to a specific target.

For example, we can center and zoom the viewport to the first potentially suspect target using it's RA/Dec from our sample table:

In [ ]:
ind = 0
mast_aladin.target = SkyCoord(
    sample["raMean"][whsuspect][ind],
    sample["decMean"][whsuspect][ind],
    unit=u.deg,
)
mast_aladin.fov = 30*u.arcsec.to(u.deg)  # FOV in degrees

# mast_aladin.aid.set_viewport(
#     center=SkyCoord(
#         sample["raMean"][whsuspect][ind],
#         sample["decMean"][whsuspect][ind],
#         unit=u.deg,
#     ),
#     fov=30*u.arcsec,
# )

We can also iterate through a subset of these suspect objects using a for loop going to every object together with a short time delay, as follows:

In [ ]:
# Change viewport location for set of suspect objects:
for ind in range(20):
    mast_aladin.target = SkyCoord(
        sample["raMean"][whsuspect][ind],
        sample["decMean"][whsuspect][ind],
        unit=u.deg
    )
    # mast_aladin.aid.set_viewport(
    #     center=SkyCoord(
    #         sample["raMean"][whsuspect][ind],
    #         sample["decMean"][whsuspect][ind],
    #         unit=u.deg,
    #     )
    # )
    time.sleep(2)

(It is also possible to delay until a key input by using instead, e.g., `input("Press any key to continue")`).


Flipping through the suspect objects, we see that these very red or blue targets include objects that appear to be very faint and/or irregular, or have very extreme colors (which may indicate cosmic rays or hot pixels in one or more filters); that are in a saturated region (and thus the colors may be unreliable); and also some objects that could indeed be real.


------

<a id="FlagChecks"></a>
### Examining flag information in the PanSTARRS catalogs
To avoid throwing out real stars in these very red/blue regions, we'll now examine the PanSTARRS catalog columns and flagging flag documentation to see if there are relevant quality cuts that can be made.

From the [Mean object table columns](https://outerspace.stsci.edu/display/PANSTARRS/PS1+MeanObjectView+table+fields) (also as seen in the [MAST catalog schema browser](https://mast.stsci.edu/schema_browser/#/), searching for "flag"), we see there is a `qualityFlag` column which contains a bitflag. The description for the flags in each bit is described in the documentation for the [PanSTARRS/PS1 object flags](https://outerspace.stsci.edu/display/PANSTARRS/PS1+Object+Flags).

There are also flag columns for each of the PanSTARRS filters, including `zFlags`, `rFlags`, `iFlags`.

We construct a quick list of all `MeanObject` table columns containing 
the word "flag" (case-insensitive)

In [ ]:
# Get a quick list of all mean table columns containing "flag" (case-insensitive)
flags = []
for key in sample.keys():
    if "flag" in key.lower():
        flags.append(key)
print(flags)

Using this column name list, we inspect the values of these flags for our suspect objects, also displaying the "objName" column.

In [ ]:
sample[whsuspect][
    "objName",
    *flags
]

While the flag description [documentation](https://outerspace.stsci.edu/display/PANSTARRS/PS1+Object+Flags) provides details on the flag values and descriptions, we will also create an instance of a bitflag "helper" class `PanSTARRSFlagHelper` --- which will help us interpret the bitflag values of individual entries, and create masks to select objects with a specific flag (or sets of flags). 

(This "helper" class is defined in the accompanying python file with this notebook.)

We first obtain the flag information for the`qualityFlag` column, which is available in the "ObjectQualityFlags" table via the [MAST PS1 DR2 TAP service](https://mast.stsci.edu/vo-tap/api/v0.1/ps1dr2).

In [ ]:
objqalflags = PanSTARRSFlagHelper(
    tap_url="https://mast.stsci.edu/vo-tap/api/v0.1/ps1dr2", 
    table="ObjectQualityFlags", 
)

Let's examine the flag definition info table from this helper:

In [ ]:
objqalflags.flag_info_table

These look quite promising as far as narrowing down our sample!

The helper class has a method to expand a bitflag into a table, with separate columns for each flag. We'll use this to expand the `qualityFlag` entries for our suspect objects to investigate whether some of these flags can be used to refine our sample.

In [ ]:
objqalflags.expand_to_table(
    sample["qualityFlag"][whsuspect], 
    ids=sample["objName"][whsuspect],
)

Using this flag-expanded table, we see that PanSTARRS quality flags that seem relevant, such as `QF_OBJ_GOOD` (False = bad) or `QF_OBJ_SUSPECT_STACK` or `QF_OBJ_BAD_STACK` (True = suspect/bad), 
appear to have "good" values for many of these potentially suspect objects, so these won't be particularly useful in this case.

Instead, data quality issues for the individual bands may be driving these outliers. Thus, we'll next examine the flags for the individual bands: `zFlags`, `rFlags`, `iFlags` 

From the documentation for the [PanSTARRS/PS1 object flags](https://outerspace.stsci.edu/display/PANSTARRS/PS1+Object+Flags), we see this flag information is in the "ObjectFilterFlags" metadata table.

We then create a new helper class instance from this metadata, and inspect the list of flags and their descriptions:

In [ ]:
filterflags = PanSTARRSFlagHelper(
    tap_url="https://mast.stsci.edu/vo-tap/api/v0.1/ps1dr2", 
    table="ObjectFilterFlags", 
)

We'll display the flag descriptions in two chunks as there are 23 total flags.

In [ ]:
# Split up as there are many flag entries:
filterflags.flag_info_table[0:10]

In [ ]:
filterflags.flag_info_table[10:23]

We note that there are flags about whether or not synthetic photometry was used (`SECF_USE_SYNTH`) 
and whether PS1 stack photometry exists (`SECF_HAS_PS1`) or whether the stack best measurement is detected and not forced `SECF_STACK_BESTDET`.  There is also a `BAD_STACK` flag.

As above, we'll use this flag helper class to expand the flags for the `z`, `r`, `i` bands. This time we'll expand only the flags we've noted above. 

We'll finish be aggregating the expanded columns into a single table for side-by-side inspection.

In [ ]:
tfilts = None
filts = ['z', 'r', 'i']
for filt in filts:
    if tfilts is None:
        ids = sample["objName"][whsuspect]
    else:
        ids = None

    # Get expanded bitflag table for each band
    tabf = filterflags.expand_to_table(
        sample[f"{filt}Flags"][whsuspect], 
        ids=ids,
        flags_to_include=["SECF_USE_SYNTH",
                          "SECF_HAS_PS1",
                          "SECF_STACK_BESTDET",
                          "BAD_STACK"],
    )

    # Create an aggregate table: list the "id":
    if tfilts is None:
        tfilts = Table({"id": tabf['id']})
        
    # Add the band/filter's flags to the aggregate table, 
    # with the filter name prepended:
    for key in tabf.keys():
        if key != "id":
            tfilts[f"{filt}_{key}"] = tabf[key]

# Reorder keys to list the same flag type together:
keys_out = ["id"]
for key in tabf.keys():
    for filt in filts:
        keys_out.append(f"{filt}_{key}")

tfilts = tfilts[keys_out]

tfilts

Leveraging this breakdown, we now print aggregate counts for each flag with the number of objects with that flag set to True/False.

In [ ]:
for key in tfilts.keys():
    if key != "id":
        strout = f"{key:>20}:   True: {np.sum(tfilts[key]):>3}"
        strout += f",  False: {np.sum(~tfilts[key]):>3}"
        print(strout)

We see that none of the suspect objects are marked as "bad" stacks, that some of the stack values are forced (and not detected, suggesting the objects are very faint in that filter), and also a decent number have *no* PS1 data & instead have synthetic data.


Revisiting the [schema for the "mean" table](https://outerspace.stsci.edu/display/PANSTARRS/PS1+MeanObjectView+table+fields), we note that there are columns `nz`, `nr`, `ni`, that contain the number of single-epoch detections in each filter.

In [ ]:
sample[whsuspect]["objName", "nz", "nr", "ni"]

By a quick aggregate count, we see that quite a few of these outliers (65 total) have either 0 or only 1 `r` detection, and some for `z` and `i` as well.

In [ ]:
for filt in filts:
    for val in [0, 1]:
        strout = f"n{filt}: number = {val}: "
        strout += f"{len(np.where(sample[whsuspect][f'n{filt}'] == val)[0])}"
        print(strout)

With only 1 detection, the magnitudes in those bands could be driven by data/noise artifacts, while those with 0 detections could be impacted by issues in the synthetic magnitudes derived from other data.

------

<a id="RefiningSample"></a>

### Refining the sample with data quality cuts

As `astroquery.mast` currently doesn't support bitflag-based filtering on only a subset of bits, we'll furst examine whether restriting our sample to only objects with **$\geq 2$ detections in each band** is sufficient to 
resolve these suspect outliers.

We make a mask to select only the subset of our sample with >= 2 detections in all 3 bands. 

In [ ]:
whmultdet = (
    (sample['nz'] >= 2)
    & (sample['nr'] >= 2)
    & (sample['ni'] >= 2)
)

We then replot our sample, marking the suspect outliers, in two ways:
* Showing all objects in the preliminary sample, with objects *without* at least 2 detections in all three bands outlined in yellow (left panel).
* Showing *only* objects with at least 2 detections in all 3 bands, with color-outlier objects marked in red as before (right panel).

In [ ]:
# Rough selection of suspect targets
f, axes = plt.subplots(1, 2)
f.set_size_inches((8.8, 4))

ax = axes[0]
ax.set_title("Preliminary sample")
ax.scatter(
    (sample['rMeanPSFMag']-sample['iMeanPSFMag']),
    sample['zMeanPSFMag'], 
    s=5, lw=0, 
    alpha=0.2
)

ax.scatter(
    (sample['rMeanPSFMag']-sample['iMeanPSFMag'])[whsuspect],
    sample['zMeanPSFMag'][whsuspect], 
    s=5, lw=0, 
    alpha=0.5,
    color='red'
)

ax.scatter(
    (sample['rMeanPSFMag']-sample['iMeanPSFMag'])[~whmultdet],
    sample['zMeanPSFMag'][~whmultdet], 
    s=7, lw=1, 
    alpha=0.5,
    edgecolor='yellow', 
    zorder=-10
)


ax.set_xlabel("r-i, PSF Mag")
ax.set_ylabel("z, PSF Mag")

ax.set_ylim(ax.get_ylim()[::-1])

xlim = ax.get_xlim()
ylim = ax.get_ylim()


ax = axes[1]
ax.set_title(r"Sample with $\geq2$ detections in all bands")
ax.scatter(
    (sample['rMeanPSFMag']-sample['iMeanPSFMag'])[whmultdet],
    sample['zMeanPSFMag'][whmultdet], 
    s=5, lw=0, 
    alpha=0.2
)

ax.scatter(
    (sample['rMeanPSFMag']-sample['iMeanPSFMag'])[whmultdet & whsuspect],
    sample['zMeanPSFMag'][whmultdet & whsuspect], 
    s=5, lw=0, 
    alpha=0.5,
    color='red'
)


ax.set_xlabel("r-i, PSF Mag")
ax.set_ylabel("z, PSF Mag")

ax.set_xlim(xlim)
ax.set_ylim(ylim)

Many of our "color-outlier" suspect targets --- as well as some others we hadn't flagged --- are removed by requiring more than one detection in each relevant band!  The resulting sample is much closer to a clean color magnitude diagram.

These constraints are something that could have be applied as part of the initial query.

Further, similar checks can reveal whether an even more refined sample could be obtained by filtering on the information from the flags explored here, or if additional flags from, e.g., the individual detection catalog. This is left as an exercise for interested readers!

------------

## Additional Resources


### Astroquery.mast

- Access to MAST holdings, including catalogs, through the [Astropy-affiliated astroquery package](https://astroquery.readthedocs.io/en/latest/mast/mast.html).
- A full list of available MAST Astroquery catalog services is available through the [astroquery documentation](https://astroquery.readthedocs.io/en/latest/mast/mast_catalog.html).


### PanSTARRS 1 DR 2

- [Documentation for the PanSTARRS Catalogs](https://outerspace.stsci.edu/display/PANSTARRS/), including additional detection informationCatalog for PanSTARRS with additional Detection information

------------
## Citations
If you use `astropy` for published research, please cite the
authors. Follow these links for more information about citing `astropy`:

* [Citing `astropy`](https://www.astropy.org/acknowledging.html)

If you use PanSTARRS data accessed through MAST for published research, 
please include the following acknowledgements, found at the following links:

* [Acknowledging PanSTARRS](https://archive.stsci.edu/publishing/mission-acknowledgements#section-895d38a0-86b3-4143-b521-6cc3312701f9)
* [Acknowledging MAST](https://archive.stsci.edu/gsc/mast_data_use.html)

## About This Notebook

The use of MAST-Aladin in this notebook draws from the demonstrations in ["Intro to MastAladinLite" by Hatice Karatay](https://github.com/haticekaratay/roman_notebooks/blob/mast-aladin-demo/content/notebooks/data_visualization/Intro_To_Mast_Aladin_Lite.ipynb).


**Authors:** Sedona Price<br>
**Keywords:** Tutorial, astroquery, MAST-Aladin, bitflags<br>
**Last updated:** June 2026 <br>

***
[Top of Page](#top)
<img style="float: right;" src="https://raw.githubusercontent.com/spacetelescope/style-guides/master/guides/images/stsci-logo.png" alt="Space Telescope Logo" width="200px"/> 